# <font color="steelblue">20. Azar y probabilidad: experimentos, sucesos y simulación</font>

*Unidad 4. Probabilidad* · Sesión teórica 20 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`municipios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["municipios.csv"]
PREGUNTAS = ["q_P20_1.json", "q_P20_2.json", "q_P20_3.json", "q_P20_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«El pueblo con más cáncer de España»: cada año aparece un titular así sobre algún municipio pequeño. Muchas veces no hay ninguna causa: en poblaciones pequeñas **el azar produce valores extremos**. Entender el azar es la base para no confundir casualidad con noticia y, más adelante, para calcular márgenes de error.

**Qué necesitas saber antes**: los cuadernos 2 a 5.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Definir experimento aleatorio, suceso y probabilidad.
2. Calcular probabilidades con la regla de Laplace y las reglas del complementario y de la unión.
3. Simular experimentos aleatorios con NumPy.
4. Explicar la ley de los grandes números y su lado periodístico: los extremos de las muestras pequeñas.

## <font color='steelblue'>Contenidos</font>

1. Experimentos, sucesos y probabilidad
2. Simular el azar
3. La ley de los grandes números
4. El lado oscuro: los extremos de lo pequeño

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Experimentos, sucesos y probabilidad</font>

- Un **experimento aleatorio** es aquel cuyo resultado no se puede predecir con certeza (lanzar un dado, elegir al azar a una persona para una encuesta).
- Un **suceso** es un conjunto de resultados («sale par», «la persona elegida se informa por redes»).
- La **probabilidad** de un suceso es un número entre 0 (imposible) y 1 (seguro).

Si todos los resultados son **igual de probables**, se usa la **regla de Laplace**:

$$P(A) = \frac{\text{casos favorables}}{\text{casos posibles}}$$

Reglas básicas:

- **Complementario**: $P(\text{no } A) = 1 - P(A)$.
- **Unión** de sucesos que no pueden ocurrir a la vez: $P(A \text{ o } B) = P(A) + P(B)$.
- **Unión** en general: $P(A \text{ o } B) = P(A) + P(B) - P(A \text{ y } B)$.

In [ ]:
# Elegimos al azar a una persona de una encuesta con estos recuentos
recuento = {"Televisión": 418, "Prensa digital": 294, "Redes sociales": 264, "Radio": 121, "Prensa en papel": 103}
total = sum(recuento.values())
p_redes = recuento["Redes sociales"] / total
p_prensa = (recuento["Prensa digital"] + recuento["Prensa en papel"]) / total
print(f"P(redes) = {p_redes:.3f}")
print(f"P(no redes) = {1 - p_redes:.3f}")
print(f"P(prensa, digital o papel) = {p_prensa:.3f}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P20_1.json")

## <font color='steelblue'>2. Simular el azar</font>

Cuando un cálculo es difícil, se puede **simular**: repetir el experimento muchas veces con el ordenador y contar. NumPy tiene un generador de números aleatorios; fijar una **semilla** (`seed`) hace que los resultados se puedan repetir.

In [ ]:
rng = np.random.default_rng(seed=1)
dado = rng.integers(1, 7, size=10)      # 10 lanzamientos de un dado
print(dado)
print("Proporción de seises:", (dado == 6).mean())

Con 10 lanzamientos la proporción de seises puede alejarse mucho de 1/6 ≈ 0,167. ¿Y con más?

In [ ]:
for n in [10, 100, 1_000, 10_000, 100_000]:
    tiradas = rng.integers(1, 7, size=n)
    print(f"n = {n:>7}: proporción de seises = {(tiradas == 6).mean():.4f}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P20_2.json")

## <font color='steelblue'>3. La ley de los grandes números</font>

La **ley de los grandes números** dice que, al repetir un experimento muchas veces, la **frecuencia relativa** de un suceso se acerca a su **probabilidad**. Es lo que hace fiables las encuestas grandes.

In [ ]:
n = 5000
tiradas = rng.integers(1, 7, size=n)
frec_acumulada = np.cumsum(tiradas == 6) / np.arange(1, n + 1)
plt.plot(frec_acumulada, color="steelblue")
plt.axhline(1 / 6, color="firebrick", ls="--", label="Probabilidad 1/6")
plt.xscale("log")
plt.xlabel("Número de lanzamientos (escala logarítmica)")
plt.ylabel("Proporción de seises")
plt.legend()
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P20_3.json")

## <font color='steelblue'>4. El lado oscuro: los extremos de lo pequeño</font>

**Qué vamos a hacer**: imaginar que en todos los municipios la probabilidad de una enfermedad es **exactamente la misma** (1 por 1.000 habitantes al año). Simulamos los casos y miramos qué municipios salen con las tasas más altas.

In [ ]:
muni = pd.read_csv("municipios.csv")
p = 0.001
muni["casos"] = rng.binomial(muni["poblacion"], p)          # azar puro: misma probabilidad en todos
muni["tasa_por_1000"] = muni["casos"] / muni["poblacion"] * 1000
muni.sort_values("tasa_por_1000", ascending=False)[["municipio", "poblacion", "casos", "tasa_por_1000"]].head(5)

In [ ]:
plt.scatter(muni["poblacion"], muni["tasa_por_1000"], alpha=0.6)
plt.axhline(1, color="firebrick", ls="--", label="Tasa real en todos: 1 por 1.000")
plt.xscale("log")
plt.xlabel("Población (escala logarítmica)")
plt.ylabel("Tasa simulada por 1.000")
plt.legend()
plt.title("Solo azar: los extremos aparecen en los municipios pequeños")
plt.show()

**Qué ha pasado**: aunque el riesgo es idéntico en todas partes, los municipios pequeños dan tasas muy altas (o cero), y los grandes se quedan cerca de 1. Esto es la **ley de los pequeños números**. Antes de titular «el pueblo con más...», mira su **población** y piensa si el azar basta para explicarlo.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P20_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Con el diccionario `recuento`, calcula la probabilidad de que una persona elegida al azar **no** se informe ni por televisión ni por radio.

<details><summary>Ver solución</summary>

```python
1 - (recuento["Televisión"] + recuento["Radio"]) / total
```
</details>

### **Ejercicio 2** *(sección 2)*

Simula 10.000 lanzamientos de dos dados y estima la probabilidad de que la suma sea 7. Compárala con el valor exacto (6/36).

<details><summary>Ver solución</summary>

```python
d1 = rng.integers(1, 7, 10000)
d2 = rng.integers(1, 7, 10000)
print(((d1 + d2) == 7).mean(), 6 / 36)
```
</details>

### **Ejercicio 3** *(sección 3)*

Simula una encuesta en la que el 30 % de la población prefiere un medio: elige al azar 50 personas y luego 5.000 (`rng.random(n) < 0.3`). ¿Cuál se acerca más al 30 %?

<details><summary>Ver solución</summary>

```python
for n in [50, 5000]:
    print(n, (rng.random(n) < 0.3).mean())
```
</details>

### **Ejercicio 4** *(sección 4)*

Repite la simulación de la sección 4 cambiando la semilla. ¿Son los mismos municipios los que salen en cabeza?

<details><summary>Ver solución</summary>

```python
rng2 = np.random.default_rng(99)
muni["casos2"] = rng2.binomial(muni["poblacion"], p)
(muni["casos2"] / muni["poblacion"] * 1000).sort_values(ascending=False).head()
```

No: cambian cada vez. Si un municipio fuera de verdad distinto, saldría en cabeza año tras año.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Un **experimento aleatorio** no tiene resultado predecible; la **probabilidad** mide entre 0 y 1 lo posible que es un suceso.
- **Laplace**: casos favorables / casos posibles (si todos son igual de probables). Complementario: 1 − P(A).
- Se puede **simular** el azar con NumPy; fija la semilla para poder repetirlo.
- **Ley de los grandes números**: con muchas repeticiones, la frecuencia relativa se acerca a la probabilidad.
- En **muestras o poblaciones pequeñas** el azar produce extremos: no los confundas con noticias.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Experimento aleatorio** | Proceso cuyo resultado no se puede predecir con certeza. |
| **Suceso** | Conjunto de resultados de un experimento. |
| **Regla de Laplace** | Probabilidad = favorables / posibles, con resultados equiprobables. |
| **Ley de los grandes números** | La frecuencia relativa tiende a la probabilidad al aumentar las repeticiones. |
| **Semilla** | Número que inicializa el generador aleatorio para poder repetir la simulación. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 21 verás la probabilidad condicionada y uno de los errores más graves en información judicial: la falacia del fiscal.

**Siguiente cuaderno**: [21. Probabilidad condicionada e independencia](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P21_probabilidad_condicionada.ipynb)